In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
model = buildBcsCnnModel(inputShape=(512, 512, 3))
model.load_weights('/content/drive/MyDrive/Equipo 5/4. Modeling/Checkpoints/bcsCnnWeights.weights.h5')

In [ ]:
testDataset = tf.keras.utils.image_dataset_from_directory(
    imagesDir,
    labels='inferred',
    label_mode='int',
    class_names=classNames,
    color_mode='rgb',
    batch_size=batchSize,
    image_size=imageSize,
    shuffle=False,
)

In [ ]:
def putRegressionLabels(images, classIndices):
    labels = tf.gather(bcsLookupTable, classIndices)
    return images, tf.expand_dims(labels, -1)

testDataset = testDataset.map(putRegressionLabels).map(onlyScale)

In [ ]:
imagenes = []
etiquetas_reales = []

for batch_images, batch_labels in testDataset:
    imagenes.append(batch_images.numpy())
    etiquetas_reales.append(batch_labels.numpy())

imagenes = np.concatenate(imagenes, axis=0)
etiquetas_reales = np.concatenate(etiquetas_reales, axis=0).flatten()

predicciones = model.predict(testDataset).flatten()
errores = np.abs(predicciones - etiquetas_reales)

In [ ]:
N = 10
indices_peores = np.argsort(errores)[-N:]

for i in indices_peores:
    plt.figure(figsize=(5, 5))
    plt.imshow(imagenes[i])
    plt.title(f"Real: {etiquetas_reales[i]:.1f} | Pred: {predicciones[i]:.2f} | Error: {errores[i]:.2f}")
    plt.axis('off')
    plt.show()